# Part 5: Continuous Batching (Problems 034–042)

**The static batching problem:**

Traditional inference groups requests into fixed batches. Once a batch starts, it runs until **every sequence in the batch finishes**. Short sequences waste GPU cycles waiting for long ones:

```
Static batching (batch_size=3):
  Seq A: ████░░░░░░░░░░░░  (done at step 4, but waits)
  Seq B: ████████████████  (done at step 16)
  Seq C: ██████░░░░░░░░░░  (done at step 6, but waits)
  GPU  : [A+B+C][A+B+C][A+B+C][A+B+C][..B..][..B..] ...
                                        ^ A,C done; GPU slots wasted
```

**Continuous batching** adds new requests to the batch as soon as a slot opens:

```
Continuous batching:
  Step 1: [A, B, C]   → A finishes at step 4
  Step 5: [D, B, C]   → D takes A's slot immediately
  Step 7: [D, B, E]   → C finishes, E takes its slot
  ...      no wasted steps!
```

## Cell 1: Simulate static batching — show wasted GPU steps

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Make sure the project root is on sys.path so solutions/ is importable
project_root = Path('__file__').parent.parent if '__file__' in dir() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Also try the current directory's parent
for p in [Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / 'solutions').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break


In [ ]:
import importlib
import random

try:
    _m = importlib.import_module("solutions.034_understand_static_batching_problem")
    simulate_static_batching = getattr(_m, "simulate_static_batching", None)
    if simulate_static_batching is None:
        funcs = [v for k, v in vars(_m).items() if callable(v) and not k.startswith('_')]
        simulate_static_batching = funcs[0] if funcs else None
except Exception:
    print("Solve problem 034 first:")
    print("  cp problems/034_understand_static_batching_problem.py solutions/034_understand_static_batching_problem.py")
    simulate_static_batching = None

random.seed(77)

# Simulate a batch of 4 sequences with different generation lengths
batch_sequences = {
    "seq_A": 5,   # short
    "seq_B": 18,  # long
    "seq_C": 8,   # medium
    "seq_D": 3,   # very short
}

max_len = max(batch_sequences.values())
total_useful_steps = sum(batch_sequences.values())
total_static_steps = len(batch_sequences) * max_len
wasted = total_static_steps - total_useful_steps

print("Static batching simulation:")
print(f"  Batch size: {len(batch_sequences)}")
print()

for seq, length in batch_sequences.items():
    bar = "#" * length + "." * (max_len - length)
    done_at = length
    wait = max_len - length
    print(f"  {seq}: [{bar}]  done at step {done_at}, wasted {wait} steps")

print()
print(f"  Total steps executed : {total_static_steps}")
print(f"  Useful compute       : {total_useful_steps}  ({100*total_useful_steps//total_static_steps}%)")
print(f"  Wasted compute       : {wasted}  ({100*wasted//total_static_steps}%)")

## Cell 2: Build a request queue with 5 sample requests

In [ ]:
import importlib

try:
    _m = importlib.import_module("solutions.035_build_request_queue")
    build_request_queue = _m.build_request_queue
except Exception:
    print("Solve problem 035 first:")
    print("  cp problems/035_build_request_queue.py solutions/035_build_request_queue.py")
    build_request_queue = None

# Sample requests
sample_requests = [
    {"request_id": "req_001", "prompt": "What is machine learning?", "max_tokens": 50},
    {"request_id": "req_002", "prompt": "Explain the transformer architecture", "max_tokens": 100},
    {"request_id": "req_003", "prompt": "Write a poem about the ocean", "max_tokens": 80},
    {"request_id": "req_004", "prompt": "Summarise: AI is changing the world", "max_tokens": 30},
    {"request_id": "req_005", "prompt": "Hello", "max_tokens": 10},
]

if build_request_queue is not None:
    queue = build_request_queue(sample_requests)
    print(f"Request queue created: {type(queue).__name__}")
    print(f"Queue size: {len(queue)}")
    print()
    print("Requests in queue:")
    for i, req in enumerate(queue):
        prompt_preview = req.get('prompt', str(req))[:40]
        print(f"  [{i}] {req.get('request_id', 'unknown')}: '{prompt_preview}...' "
              f"max_tokens={req.get('max_tokens', '?')}")
else:
    print("Requests that would be queued:")
    for req in sample_requests:
        print(f"  {req['request_id']}: '{req['prompt'][:40]}' max_tokens={req['max_tokens']}")

## Cell 3: Run the iteration-level scheduler for a few steps

In [ ]:
import importlib

try:
    _m = importlib.import_module("solutions.037_iteration_level_scheduler")
    iteration_level_scheduler = _m.iteration_level_scheduler
except Exception:
    print("Solve problem 037 first:")
    print("  cp problems/037_iteration_level_scheduler.py solutions/037_iteration_level_scheduler.py")
    iteration_level_scheduler = None

sample_requests = [
    {"request_id": f"req_{i:03d}", "prompt": f"prompt {i}", "max_tokens": (i % 5) + 3}
    for i in range(1, 9)
]

if iteration_level_scheduler is not None:
    scheduler = iteration_level_scheduler(
        requests=sample_requests,
        max_batch_size=3,
    )

    print("Running scheduler for 10 steps:")
    print(f"  {'Step':>5}  {'Running':>30}  {'Waiting':>8}")
    print(f"  {'-'*5}  {'-'*30}  {'-'*8}")

    for step in range(10):
        state = next(scheduler, None)
        if state is None:
            print(f"  {step:>5}  [all done]")
            break
        running = [r.get('request_id', str(r)) for r in state.get('running', [])]
        waiting = len(state.get('waiting', []))
        print(f"  {step:>5}  {str(running):<30}  {waiting:>8} waiting")
else:
    print("Implement problem 037 to see the iteration-level scheduler in action.")

## Cell 4: Run the full continuous batching loop on synthetic requests

In [ ]:
import importlib
import time

try:
    _m = importlib.import_module("solutions.041_run_continuous_batching_loop")
    run_continuous_batching_loop = _m.run_continuous_batching_loop
except Exception:
    print("Solve problem 041 first:")
    print("  cp problems/041_run_continuous_batching_loop.py solutions/041_run_continuous_batching_loop.py")
    run_continuous_batching_loop = None

# Generate 10 synthetic requests
import random
random.seed(5)
requests = [
    {"request_id": f"req_{i:03d}", "prompt": f"sample prompt number {i}",
     "max_tokens": random.randint(5, 25)}
    for i in range(10)
]

if run_continuous_batching_loop is not None:
    print("Running continuous batching loop on 10 synthetic requests...")
    start = time.perf_counter()
    results = run_continuous_batching_loop(
        requests=requests,
        max_batch_size=4,
    )
    elapsed = time.perf_counter() - start

    print(f"\nCompleted {len(results)} requests in {elapsed:.3f}s")
    print()
    print(f"  {'Request':>10}  {'Tokens':>8}  {'Generated text (preview)':>30}")
    print(f"  {'-'*10}  {'-'*8}  {'-'*30}")
    for res in results[:5]:
        text_preview = str(res.get('generated_text', res))[:30]
        tokens = res.get('tokens_generated', '?')
        print(f"  {res.get('request_id', 'unknown'):>10}  {str(tokens):>8}  {text_preview}")
    if len(results) > 5:
        print(f"  ... and {len(results)-5} more")
else:
    print("Implement problem 041 to run the full continuous batching loop.")

## Cell 5: Benchmark throughput — continuous vs static batching

In [ ]:
import importlib
import matplotlib
import matplotlib.pyplot as plt

try:
    _m = importlib.import_module("solutions.042_benchmark_throughput")
    benchmark_throughput = _m.benchmark_throughput
except Exception:
    print("Solve problem 042 first:")
    print("  cp problems/042_benchmark_throughput.py solutions/042_benchmark_throughput.py")
    benchmark_throughput = None

if benchmark_throughput is not None:
    results = benchmark_throughput(n_requests=20, max_batch_size=4)
    print("Throughput benchmark:")
    print(f"  Static batching   : {results['static_throughput']:.2f} tokens/sec")
    print(f"  Continuous batching: {results['continuous_throughput']:.2f} tokens/sec")
    print(f"  Speedup: {results['speedup']:.2f}x")

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(
        ["Static Batching", "Continuous Batching"],
        [results['static_throughput'], results['continuous_throughput']],
        color=["lightcoral", "steelblue"]
    )
    ax.set_ylabel("Throughput (tokens/sec)")
    ax.set_title("Continuous vs Static Batching Throughput")
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height * 1.01,
                f'{height:.1f}', ha='center', va='bottom')
    plt.tight_layout()
    plt.savefig("/tmp/batching_throughput.png", dpi=100)
    plt.show()
else:
    # Theoretical illustration
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Static", "Continuous"], [1.0, 2.5], color=["lightcoral", "steelblue"])
    ax.set_ylabel("Relative Throughput")
    ax.set_title("Expected speedup from continuous batching\n[solve problem 042 for real numbers]")
    plt.tight_layout()
    plt.show()
    print("Continuous batching typically achieves 2-5x higher throughput.")